# SAM Demo — Workspace Setup

This notebook builds the complete Simulated Asset Management demo environment step by step.
Run each cell individually to control what gets created, or use **Run All** for the full build (~15-20 min).

**Prerequisites:**
1. Run `scripts/setup.sql` in a worksheet (creates DB, role, warehouses, grants)
2. Connect a notebook service (Python 3.11+, any compute pool)
3. Install packages: open Terminal and run `pip install -r "$PWD/requirements.txt"`
4. Restart the kernel after installing packages

In [ ]:
import os
import sys
from datetime import datetime

os.environ['SNOWFLAKE_NOTEBOOK_RUNTIME'] = '1'

# Resolve workspace root (parent of notebooks/)
_cwd = os.getcwd()
_python_dir = os.path.join(_cwd, 'python')
if not os.path.isdir(_python_dir):
    # Might be running from inside notebooks/ — go up one level
    _python_dir = os.path.join(os.path.dirname(_cwd), 'python')
if not os.path.isdir(_python_dir):
    # Fallback: workspace root symlink
    _python_dir = '/workspace/python' if os.path.isdir('/workspace/python') else _cwd
if _python_dir not in sys.path:
    sys.path.insert(0, _python_dir)

from snowflake.snowpark.context import get_active_session
import config
from utils.logging import (
    log_phase, log_step, log_substep, log_detail,
    log_warning, log_phase_complete, set_verbosity
)

set_verbosity(2)
session = get_active_session()

db_name = config.DATABASE['name']
wh_name = config.WAREHOUSES['execution']['name']

session.sql("USE ROLE LAB_ASSETAGENT_ADMIN").collect()
session.sql(f"USE DATABASE {db_name}").collect()
session.sql(f"USE WAREHOUSE {wh_name}").collect()

print(f"Database:   {db_name}")
print(f"Warehouse:  {wh_name}")
print(f"Scenarios:  {', '.join(config.AVAILABLE_SCENARIOS)}")
print(f"Ready.")

## Step 1: Foundation (Dimension Tables)

Creates the database schemas and core dimension tables in `CURATED`:

| Object | Schema | Description |
|--------|--------|-------------|
| DIM_ISSUER | CURATED | 79 real issuers (Apple, Microsoft, etc.) with CIK, GICS sector |
| DIM_SECURITY | CURATED | Securities derived from issuers (ticker, CUSIP, asset class) |
| DIM_PORTFOLIO | CURATED | 11 portfolios (growth, value, ESG, thematic strategies) |
| DIM_BENCHMARK | CURATED | 3 benchmarks (S&P 500, MSCI ACWI, Nasdaq 100) |
| DIM_SUPPLY_CHAIN_RELATIONSHIPS | CURATED | Supply chain network between issuers |
| DIM_PORTFOLIO_IPS | CURATED | Investment Policy Statements with risk constraints |

In [ ]:
from data import structured as generate_structured
from data import market_data as generate_market_data

generate_structured.create_database_structure(session, recreate_database=False)
generate_structured.build_dimension_tables(session, test_mode=False)

print("Step 1 complete: dimension tables created.")

## Step 2: Market Data (Marketplace)

Loads real financial data from the Cybersyn Marketplace listing into `MARKET_DATA`:

| Object | Schema | Description |
|--------|--------|-------------|
| FACT_STOCK_PRICES | MARKET_DATA | ~98K rows of daily stock prices for all securities |
| FACT_SEC_FINANCIALS | MARKET_DATA | ~7.5K rows of SEC financial statements |
| FACT_SEC_SEGMENTS | MARKET_DATA | Business segment revenue breakdowns |
| FACT_ESTIMATE_CONSENSUS | MARKET_DATA | ~5.6K analyst estimate consensus records |
| FACT_TREASURY_YIELDS | MARKET_DATA | 14 maturities of daily yield curve |
| FACT_COUNTRY_EMISSIONS | MARKET_DATA | 190+ countries emissions data |
| FACT_DIVIDENDS | MARKET_DATA | Dividend payments extracted from SEC 8-K filings |
| DIM_BROKER, DIM_ANALYST | MARKET_DATA | Broker and analyst reference data |
| + 5 more | MARKET_DATA | Policy rates, FX rates, economic indicators, insider/institutional holdings |

In [ ]:
from data import market_data as generate_market_data

generate_market_data.build_price_anchor(session, test_mode=False)
generate_market_data.build_fact_dividends(session, test_mode=False)
generate_market_data.build_all(session, test_mode=False)

print("Step 2 complete: market data loaded.")

## Step 3: Curated + Analytics

Builds derived fact tables, views, and attribution analytics in `CURATED`:

| Category | Objects Created |
|----------|----------------|
| **Core facts** | FACT_TRANSACTION, FACT_POSITION_DAILY_ABOR, FACT_ESG_SCORES, FACT_COMPLIANCE_ALERTS, FACT_RISK_LIMITS, etc. |
| **Client data** | DIM_CLIENT, DIM_CLIENT_MANDATES, FACT_CLIENT_FLOWS, FACT_FUND_FLOWS |
| **Operations** | FACT_TRADE_SETTLEMENT, FACT_RECONCILIATION, FACT_NAV_CALCULATION, FACT_CASH_MOVEMENTS |
| **Attribution** | FACT_BRINSON_ATTRIBUTION, FACT_FACTOR_ATTRIBUTION, FACT_CURRENCY_ATTRIBUTION, FACT_BRINSON_LINKED |
| **Modelling** | FACT_COVARIANCE_MATRIX, FACT_EXPECTED_RETURNS, DIM_MODEL_PORTFOLIO |
| **Views** | V_SECURITY_RETURNS, V_HOLDINGS_WITH_ESG, V_PORTFOLIO_BENCHMARK_COMPARISON, V_MACRO_REGIME |
| **Performance** | FACT_STRATEGY_PERFORMANCE, FACT_BENCHMARK_PERFORMANCE |

In [ ]:
from data import structured as generate_structured

generate_structured.build_fact_tables(session, test_mode=False)

generate_structured.reset_build_tracking()
for scenario in config.AVAILABLE_SCENARIOS:
    generate_structured.build_scenario_data(session, scenario)

generate_structured.validate_data_quality(session)
generate_structured.build_security_returns_view(session)
generate_structured.build_v_holdings_with_esg(session)

data_phases = config.get_data_phases(config.AVAILABLE_SCENARIOS)

if 'attribution' in data_phases:
    log_substep("Attribution market data")
    generate_structured.build_attribution_market_data(session)

if 'factor_exposures' in data_phases:
    log_substep("Factor exposures")
    generate_structured.build_factor_exposures(session)

if 'attribution' in data_phases:
    log_substep("Attribution tables")
    generate_structured.build_attribution_tables(session)
    generate_structured.build_multi_level_attribution(session)
    generate_structured.build_currency_attribution(session)
    generate_structured.build_attribution_linked(session)
    generate_structured.build_advanced_attribution_views(session)

if 'portfolio_modelling' in data_phases:
    log_substep("Portfolio modelling views")
    generate_structured.build_portfolio_modelling_views(session)
    generate_structured.build_fact_covariance_matrix(session)

generate_structured.build_fact_strategy_performance(session)
generate_structured.build_fact_benchmark_performance(session)
generate_structured.build_portfolio_benchmark_comparison_view(session)

print("Step 3 complete: curated + analytics built.")

## Step 4: Pipelines + Corpus

Creates document pipelines (stages, streams, tasks) and generates the unstructured document corpus:

| Category | Objects Created |
|----------|----------------|
| **Pipeline infra** | Stages (PDF_INTERNAL_STAGE, PDF_EXTERNAL_STAGE, etc.), streams, task DAGs in RAW |
| **RAW tables** | 15+ document RAW tables for each document type |
| **Corpus tables** | PDF_INTERNAL_CORPUS, PDF_EXTERNAL_CORPUS, PDF_REGULATORY_CORPUS + per-type corpus tables in CURATED |
| **Documents generated** | Broker research, press releases, engagement notes, policy docs, sales templates, custodian reports, reconciliation notes, SSI documents, regulatory docs |
| **Real data loaded** | SEC filings (FACT_SEC_FILING_TEXT), earnings transcripts |

Documents are hydrated from `content_library/` templates with real company data, then parsed by pipeline tasks into chunked corpus tables.

In [ ]:
from data.coverage_universe import build_coverage_universe
from data import pipelines as create_unstructured_pipelines
from utils.config_helpers import get_required_document_types

build_coverage_universe(session)
create_unstructured_pipelines.create_all_pipelines(session)

required_doc_types = get_required_document_types(config.AVAILABLE_SCENARIOS)

if 'company_event_transcripts' in required_doc_types:
    try:
        from data import transcripts as generate_real_transcripts
        if generate_real_transcripts.verify_transcripts_available(session):
            log_substep("Loading real transcripts")
            generate_real_transcripts.load_raw_table(session, test_mode=False)
    except Exception as e:
        log_warning(f"Real transcripts: {e}")

try:
    log_substep("Loading SEC filings")
    create_unstructured_pipelines.load_sec_filings_raw(session, test_mode=False)
except Exception as e:
    log_warning(f"SEC filings: {e}")

log_substep("Generating documents from templates")
from data import unstructured as generate_unstructured
generate_unstructured.build_all(session, required_doc_types, test_mode=False)

log_substep("Executing pipelines")
create_unstructured_pipelines.run_all_pipelines(session, upload_pdfs=True)

print("Step 4 complete: pipelines + corpus built.")

## Step 5: NLP Scoring + Hidden Factors

Runs AI functions on the corpus to generate NLP scores and hidden factor exposures:

| Object | Schema | Description |
|--------|--------|-------------|
| Transcript NLP scores | MARKET_DATA | AI_COMPLETE sentiment/topic scores on earnings transcripts |
| FACT_HIDDEN_FACTOR_EXPOSURES | CURATED | 5 hidden thematic factors (AI Exposure, Reshoring, Rate Convexity, Climate, Geopolitical) |

*Skipped automatically if `nlp_scoring` is not in the active data phases.*

In [ ]:
from data import structured as generate_structured
from data import market_data as generate_market_data

data_phases = config.get_data_phases(config.AVAILABLE_SCENARIOS)

if 'nlp_scoring' not in data_phases:
    print("Step 5: Skipped (nlp_scoring not in data phases)")
else:
    log_phase("Step 5: NLP Scoring + Hidden Factors")
    log_substep("Transcript NLP scores")
    generate_market_data.build_transcript_nlp_scores(session, test_mode=False)
    log_substep("Hidden factor exposures")
    generate_structured.build_hidden_factor_exposures(session)
    print("Step 5 complete: NLP scoring + hidden factors built.")

## Step 6: AI Components

Creates all Cortex AI infrastructure in the `AI` schema:

| Category | Objects Created |
|----------|----------------|
| **Infrastructure tables** | FACT_PROACTIVE_INSIGHTS, FACT_PROACTIVE_ALERTS, FACT_SIGNALS, FACT_RESEARCH_THESES |
| **Semantic Views** | ~16 views (SAM_PORTFOLIO_VIEW, SAM_ATTRIBUTION_VIEW, SAM_MARKET_VIEW, SAM_RESEARCH_VIEW, SAM_EXECUTIVE_VIEW, etc.) |
| **Cortex Search Services** | ~16 services (SAM_BROKER_RESEARCH, SAM_COMPANY_EVENTS, SAM_PRESS_RELEASES, SAM_REAL_SEC_FILINGS, etc.) |
| **Tools (UDFs/procedures)** | RUN_BACKTEST_TOOL, RUN_MONTE_CARLO_TOOL, RUN_ATTRIBUTION_TOOL, RUN_STRESS_BACKTEST_TOOL, GENERATE_PDF_REPORT, MA_SIMULATION_TOOL, etc. |
| **Skills** | 36 SKILL.md files uploaded to SKILL_STAGE |
| **Agents** | ~8 Cortex Agents registered with Snowflake Intelligence (portfolio, research, sales, executive, risk, ops, credit, PE) |
| **Evaluation datasets** | Per-agent evaluation question sets |

In [ ]:
from ai.proactive_insights import create_proactive_insights_tables
from ai.signal_store import create_fact_signals
from ai.thesis_tracker import setup_thesis_tracker
from ai import builder as build_ai

create_proactive_insights_tables(session)
create_fact_signals(session)
setup_thesis_tracker(session)

build_ai.build_all(
    session,
    config.AVAILABLE_SCENARIOS,
    build_semantic=True,
    build_search=True,
    build_agents=True,
    verify_only=False
)

print("Step 6 complete: AI components built.")

## Step 7: Signals + Briefings

Extracts investment signals and generates morning briefings:

| Object | Schema | Description |
|--------|--------|-------------|
| FACT_SIGNALS (seeded) | CURATED | Tier 1 + Tier 2 investment signals extracted from documents |
| FACT_PROACTIVE_INSIGHTS (seeded) | AI | Morning briefings for 6 personas (equity, credit, PE, executive, risk, ops) |

In [ ]:
from ai.signal_store import seed_signals
log_substep("Extracting signals")
seed_signals(session, test_mode=False, include_tier2=True, include_tier3=False)

from ai.proactive_insights import seed_morning_briefings
log_substep("Generating morning briefings")
seed_morning_briefings(session, personas=[
    'equity', 'credit', 'pe', 'executive',
    'risk-compliance', 'operations',
])

print("Step 7 complete: signals + briefings seeded.")

## Step 8: Earnings Insights

Generates AI-powered earnings insights:

| Object | Schema | Description |
|--------|--------|-------------|
| FACT_EARNINGS_INSIGHTS | AI | AI-generated insights from earnings transcripts and financial data |

In [ ]:
from ai.earnings_insights import seed_earnings_insights
seed_earnings_insights(session)

print("Step 8 complete: earnings insights seeded.")

## Step 9: Agent Evaluation Datasets

Creates evaluation question sets for testing agent quality:

| Object | Schema | Description |
|--------|--------|-------------|
| EVAL_* tables | AI | Per-agent evaluation datasets with expected questions and validation criteria |

In [ ]:
from ai.evaluations import create_eval_datasets
created, failed = create_eval_datasets(session, config.AVAILABLE_SCENARIOS)

print(f"Step 9 complete: {created} evaluation datasets created ({failed} failed).")

## Setup Complete

**Next steps:**
1. Open **Snowflake Intelligence** to interact with the agents
2. Open the ML notebooks (`factor_discovery`, `market_regime_detection`, `credit_risk_model`) for ML workflow demonstrations